# Run queued voice jobs on Colab (by hand)

Runs the jobs in `JOBS` one after another (default: today's queue, book test first) and saves each one's results to Google Drive → `voice-training-results/<job>/`. Claude collects them from Drive and commits them.

1. **Runtime → Change runtime type → T4 GPU**, save.
2. **Runtime → Run all**, and allow Google Drive access when asked.
3. Keep the tab open. Each job saves to Drive as soon as it finishes, so stopping halfway keeps the finished ones.

In [ ]:
# 1 · which jobs (edit if you like; the book test is about 25 min, each normal batch about 15 min)
JOBS = ['run18_book', 'full_normal_1', 'full_normal_2', 'full_normal_3', 'full_normal_4', 'run16_ellipsis', 'run17_low']

In [ ]:
# 2 · GPU check
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU: Runtime → Change runtime type → T4 GPU (or Colab has no free GPU for this account right now)'
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)

In [ ]:
# 3 · connect Google Drive (results are saved there)
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/voice-training-results'
os.makedirs(ROOT, exist_ok=True); print('results go to', ROOT)

In [ ]:
# 4 · get the code
!rm -rf /content/vt && git clone -q --depth 1 https://github.com/28thanujaya-hash/voice-training /content/vt && echo ok

In [ ]:
# 5 · run each job and save it to Drive as soon as it finishes
import shutil, subprocess, time
for job in JOBS:
    d = f'/content/vt/kaggle/jobs/{job}'
    if not os.path.exists(d + '/run.py'): print('skip (not found):', job); continue
    shutil.rmtree('/kaggle/working', ignore_errors=True); os.makedirs('/kaggle/working')
    t = time.time(); print(f'=== {job} ===', flush=True)
    p = subprocess.run(['python', 'run.py'], cwd=d)
    print(f'{job}: exit {p.returncode}, {(time.time()-t)/60:.0f} min', flush=True)
    out = f'{ROOT}/{job}'; shutil.rmtree(out, ignore_errors=True)
    shutil.copytree('/kaggle/working', out); print('saved to', out, flush=True)